# OLMo 2 7B GRPO RL Post-Training on GSM8K

## Introduction
This file is to implement RL post training on OLMo2 7B SFT model. I applied GRPO for training. It supports interrupted training, automatic resume, and periodically saved to the checkpoint and training state to Google Drive.

## Structure
1. **Environment Setup**
   - Install required libraries and dependencies.

2. **Imports and Global Configuration**
   - Import PyTorch, Hugging Face, TRL, PEFT, and utility libraries.
   - Define output directories, checkpoint paths, and training constants.

3. **Checkpoint and Resume System**
   - Implement automatic checkpoint saving.
   - Store completed training steps in a JSON state file.
   - Resume interrupted training from the latest checkpoint.

4. **Dataset Loading and Formatting**
   - Load the GSM8K dataset.
   - Convert examples into chat-style prompts.
   - Extract gold answers for reward computation.

5. **Answer Extraction and Reward Functions**
   - Parse model outputs to extract boxed answers.
   - Define correctness-based rewards for GRPO training.

6. **Callback System**
   - Track training progress and timing.
   - Periodically evaluate validation performance.
   - Save intermediate checkpoints safely.

7. **Model Loading and Quantization**
   - Load the OLMo 2 7B SFT model.
   - Apply 4-bit quantization to reduce GPU memory usage.

8. **LoRA Configuration**
   - Apply parameter-efficient fine-tuning to attention and MLP projection layers.
   - Reduce the number of trainable parameters.

9. **GRPO Training Configuration**
   - Configure reinforcement learning hyperparameters and load LoRA parameters.
   - Define generation settings, KL regularization, and optimization behavior.

10. **Training Execution**
    - Initialize the GRPO trainer.
    - Resume from checkpoints if available.
    - Perform reinforcement learning fine-tuning.

11. **Evaluation**
    - Run inference on GSM8K validation examples.
    - Measure reasoning accuracy and final-answer correctness.

This modular structure improves readability, reproducibility, and fault tolerance, while also enabling long-duration training across multiple Colab sessions.

## 1. Environment Setup

In [ ]:
# Install libraries and dependencies for GRPO training.
!pip install -q transformers trl peft bitsandbytes accelerate datasets
# These codes will kill the program. Ignore it and then run the following cells.
import os; os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 49.8 MB/s eta 0:00:00


## 2. Import PyTorch, Hugging Face, TRL, PEFT, and utility libraries.

In [ ]:
import os, re, json, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainerCallback
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer
from google.colab import drive
from tqdm import tqdm
import time
from transformers import TrainerCallback

## Connect with Google Drive to save the results

In [ ]:
# ── 2. Mount Drive ─────────────────────────────────────────────────────────
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/olmo2-gsm8k-grpo"
STATE_FILE  = os.path.join(OUTPUT_DIR, "run_state.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)


Mounted at /content/drive


## 3. Resume previous training states and checkpoints
You can adjust this STEPS_THIS_RUN variable to decide how many steps to run in this section.

In [ ]:
# ── 3. Run state (pause / resume) ─────────────────────────────────────────

# ── Configuration ──────────────────────────────────────────────────────────
TOTAL_STEPS    = 500   # final target — keep this the same across all sessions
STEPS_THIS_RUN = 100   # steps to run in THIS session — adjust freely

# ── Run state ──────────────────────────────────────────────────────────────
if os.path.exists(STATE_FILE):
    with open(STATE_FILE) as f:
        run_state = json.load(f)
    COMPLETED_STEPS = run_state["completed_steps"]
    print(f"Resuming from step {COMPLETED_STEPS}/{TOTAL_STEPS}")
else:
    COMPLETED_STEPS = 0
    print("Starting fresh run")

REMAINING_STEPS  = TOTAL_STEPS - COMPLETED_STEPS
STEPS_THIS_RUN   = min(STEPS_THIS_RUN, REMAINING_STEPS)
ABSOLUTE_TARGET_STEP = COMPLETED_STEPS + STEPS_THIS_RUN

print(f"Running {STEPS_THIS_RUN} steps this session "
      f"(steps {COMPLETED_STEPS+1}-{COMPLETED_STEPS+STEPS_THIS_RUN} of {TOTAL_STEPS})")

if STEPS_THIS_RUN <= 0:
    print("Training already complete — skipping to evaluation.")

Resuming from step 350/500
Running 100 steps this session (steps 351-450 of 500)


## 4. Builing prompts for training
A CoT prompt to help generate more correct answer

In [ ]:
# ── 4. Prompt ─────────────────────────────────────────────────────────
def format_example(example):
    raw_answer = example["answer"].split("####")[-1].strip()
    return {
        # Raw string prompt — not a chat template dict
        "prompt": (
            "<|im_start|>system\n"
            "You are a helpful assistant.<|im_end|>\n"
            "<|im_start|>user\n"
            f"{example['question']}\n"
            "Please reason step by step, and put your final answer within\\boxed{}.<|im_end|>\n"
            "<|im_start|>assistant\n"
        ),
        "answer": raw_answer,
    }

# Download the dataset
dataset = load_dataset("gsm8k", "main")

# Builing prompts for training
train_ds = dataset["train"].map(format_example)
test_ds  = dataset["test"].map(format_example)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

## 5. Extract numerical answer from full text answer and Reward function

In [ ]:
# ── 5. Extract Answer ─────────────────────────────────────────────────────────
def extract_boxed(text):
    """Extract content from the last \\boxed{} in the text."""
    matches = re.findall(r"\\boxed\{([^}]+)\}", text)
    return matches[-1].strip() if matches else None

def normalize_number(s):
    """Normalize numeric strings for comparison."""
    if s is None:
        return None
    s = s.replace(",", "").strip()
    try:
        return str(float(s)) if "." in s else str(int(s))
    except ValueError:
        return s.lower()

def reward_correctness(completions, answer, **kwargs):
  """The reward funtion only reward for the correct answers"""
    rewards = []
    for i, completion in enumerate(completions):
        # Handle completion as string or list of dicts
        if isinstance(completion, str):
            text = completion
        else:
            text = completion[0]["content"]

        # Handle answer as list (batched) or plain string
        gold_raw = answer[i] if isinstance(answer, list) else answer

        pred = normalize_number(extract_boxed(text))
        gold = normalize_number(gold_raw)
        rewards.append(1.0 if (pred is not None and pred == gold) else 0.0)
    return rewards

## 6. Callback function for GRPO training
including training progress, timing and validation.

In [ ]:
# ── 6. Callback Function ─────────────────────────────────────────────────
class CheckpointStateCallback(TrainerCallback):
  """Track training progress."""
    def __init__(self, state_file, total_steps):
        self.state_file  = state_file
        self.total_steps = total_steps

    def on_save(self, args, state, control, **kwargs):
        with open(self.state_file, "w") as f:
            json.dump({
                "completed_steps":   state.global_step,
                "total_target_steps": self.total_steps,
            }, f, indent=2)
        print(f"[Checkpoint] Run state saved at step {state.global_step}")



class StepTimerCallback(TrainerCallback):
  """Track training timing."""
    def __init__(self, sample_steps=5):
        self.sample_steps = sample_steps  # measure average over first N steps
        self.step_times   = []
        self.step_start   = None

    def on_step_begin(self, args, state, control, **kwargs):
        self.step_start = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if self.step_start is None:
            return
        elapsed = time.time() - self.step_start
        self.step_times.append(elapsed)
        self.step_start = None

        if len(self.step_times) == self.sample_steps:
            avg = sum(self.step_times) / self.sample_steps
            remaining = args.max_steps - state.global_step
            eta_min = avg * remaining / 60
            print(f"\n── Timing estimate ──────────────────────────")
            print(f"  Avg time/step : {avg:.1f}s  (over {self.sample_steps} steps)")
            print(f"  Steps left    : {remaining}")
            print(f"  ETA this run  : {eta_min:.0f} min  (~{eta_min/60:.1f} hrs)")
            print(f"─────────────────────────────────────────────\n")

class ValidationCallback(TrainerCallback):
  """Periodically evaluate validation performance."""
    def __init__(self, model, tokenizer, val_dataset,
                 eval_every=50, n_samples=50):
        self.model       = model
        self.tokenizer   = tokenizer
        self.val_dataset = val_dataset.select(range(n_samples))
        self.eval_every  = eval_every
        self.history     = []   # tracks (step, accuracy)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.eval_every != 0:
            return
        if state.global_step == 0:
            return

        correct = 0
        self.model.eval()
        for ex in self.val_dataset:
            inputs = self.tokenizer(
                ex["prompt"], return_tensors="pt"
            ).input_ids.to(self.model.device)
            with torch.no_grad():
                out = self.model.generate(
                    inputs, max_new_tokens=512, do_sample=False
                )
            text = self.tokenizer.decode(
                out[0][inputs.shape[1]:], skip_special_tokens=True
            )
            if normalize_number(extract_boxed(text)) == \
               normalize_number(ex["answer"]):
                correct += 1

        acc = correct / len(self.val_dataset)
        self.history.append((state.global_step, acc))
        print(f"\n[Val @ step {state.global_step}] "
              f"Accuracy: {correct}/{len(self.val_dataset)} = {acc:.2%}")
        print(f"  History: {self.history}\n")
        self.model.train()

In [ ]:
# Initialize ValidationCallback for current tokenizer and model
val_callback = ValidationCallback(
    model, tokenizer, test_ds,
    eval_every=50,   # evaluate every 50 steps
    n_samples=50,    # use 50 test examples (fast, ~2-3 min)
)

## 7. Download the model

In [ ]:
# ── 7. Model ───────────────────────────────────────────────────────────────
MODEL_ID = "allenai/OLMo-2-1124-7B-SFT"

# Apply 4-bit quantization to reduce GPU memory usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Loading tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",
    dtype=torch.bfloat16,
)

# Initialize model
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/675 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

## 8. Trained LoRA parameters to reduce the number of trainable parameters

In [ ]:
# ── 8. LoRA ────────────────────────────────────────────────────────────────
# Configuration of LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## 9. Download the LoRA parameters from latest checkpoint

In [ ]:
# ── 9. Find latest LoRA checkpoint ──────────────────────────────────────────────
def get_latest_checkpoint(output_dir):
    if not os.path.exists(output_dir):
        return None
    checkpoints = [
        d for d in os.listdir(output_dir)
        if d.startswith("checkpoint-") and
           os.path.isdir(os.path.join(output_dir, d))
    ]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    return os.path.join(output_dir, latest)

resume_from = get_latest_checkpoint(OUTPUT_DIR)
print(f"Checkpoint: {resume_from or 'none — fresh start'}")

Checkpoint: /content/drive/MyDrive/olmo2-gsm8k-grpo/checkpoint-350


## 10. GRPO Training Loops

In [ ]:
# ── 9. Train ───────────────────────────────────────────────────────────────
if REMAINING_STEPS > 0:

    # GRPO Configuration
    grpo_config = GRPOConfig(
        num_generations=4,
        max_completion_length=512,

        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,

        max_steps=ABSOLUTE_TARGET_STEP,   # ← was REMAINING_STEPS
        save_steps=50,
        save_total_limit=2,

        learning_rate=5e-6,
        lr_scheduler_type="cosine",
        warmup_steps=25,

        gradient_checkpointing=True,
        bf16=True,
        optim="paged_adamw_8bit",

        logging_steps=10,
        output_dir=OUTPUT_DIR,
        report_to="none",
        temperature=0.7,
        beta=0.04,
    )

    # GRPO training
    trainer = GRPOTrainer(
        model=model,
        args=grpo_config,
        processing_class=tokenizer,
        reward_funcs=[reward_correctness],
        train_dataset=train_ds,
        peft_config=lora_config,
        callbacks=[CheckpointStateCallback(STATE_FILE, TOTAL_STEPS),
                   StepTimerCallback(sample_steps=5),
                   val_callback],
    )

    trainer.train(resume_from_checkpoint=resume_from)

    print("Session complete. Safe to disconnect.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 100257}.


Step,Training Loss
360,0.006490
370,0.024746
380,-0.038527
390,0.014050
400,0.077869
410,0.022123
420,0.098207
430,0.056525
440,0.005948
450,0.054567



── Timing estimate ──────────────────────────
  Avg time/step : 47.0s  (over 5 steps)
  Steps left    : 95
  ETA this run  : 74 min  (~1.2 hrs)
─────────────────────────────────────────────


[Val @ step 400] Accuracy: 40/50 = 80.00%
  History: [(400, 0.8)]

[Checkpoint] Run state saved at step 400

[Val @ step 450] Accuracy: 41/50 = 82.00%
  History: [(400, 0.8), (450, 0.82)]

[Checkpoint] Run state saved at step 450
Session complete. Safe to disconnect.


100

## 11. Evalution for the model after training


In [ ]:
def evaluate_gsm8k(model, tokenizer, dataset, n=20, batch_size=8):
    model.eval()

    # Set pad token if not set (needed for batching)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id

    samples  = dataset.select(range(n))
    prompts  = [ex["prompt"]  for ex in samples]
    answers  = [ex["answer"]  for ex in samples]
    correct  = 0

    for i in tqdm(range(0, n, batch_size), desc="Evaluating"):
        batch_prompts = prompts[i : i + batch_size]
        batch_answers = answers[i : i + batch_size]

        # Pad to same length, left-pad so generation continues from the right
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,      # reduced from 512 — GSM8K rarely needs more
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decode only the generated part (strip the prompt)
        for j, output in enumerate(outputs):
            prompt_len = inputs.input_ids[j].shape[0]
            text = tokenizer.decode(
                output[prompt_len:], skip_special_tokens=True
            )
            if normalize_number(extract_boxed(text)) == \
               normalize_number(batch_answers[j]):
                correct += 1

    print(f"Pass@1: {correct}/{n} = {correct/n:.2%}")

# Uncomment following line to evaluate the result
# evaluate_gsm8k(model, tokenizer, test_ds)